# Version note:
- Try out reasoning by adding "Before you respond to my query please walk me through through your thought process step by step." phrase.
- Testin tables.

In [1]:
# Import required libraries
import numpy as np
import pandas as pd
import os
import time

from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import CharacterTextSplitter, RecursiveCharacterTextSplitter
from ragas import EvaluationDataset, evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ContextRecall, LLMContextPrecisionWithReference, ResponseRelevancy


from pathlib import Path
from typing import List, Optional, Tuple

from langchain_community.vectorstores import Chroma
from langchain_ollama import ChatOllama
from langchain.schema import Document

# import fitz  # PyMuPDF
from pathlib import Path



In [2]:
import getpass
import os

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")

In [3]:
FILE_PATH = "C:/Users/Widi/rag/docs"

# DIRECTORY_PATH = "C:/Users/Widi/rag/docs/test"
DIRECTORY_PATH = "C:/Users/Widi/rag/docs/test_tables"
# GEN_MODEL_ID = "gemma3:12b"

# The model used in RAG architecture
EVAL_LLM_CONFIG = {
    "model" : "gemma3:4b",
    "temperature" : 0,
}

EVAL_EMBED_CONFIG = {
    "model" : "nomic-embed-text:latest"
}


In [4]:
# RAG architecture
class RAG:
    def __init__(
        self,
        llm_config: dict,
        embed_config: dict,
        persist_directory: Optional[str] = None,
        collection_name: str = "pirls_tables_ver_2",
        chunk_size: int = 1000,
        chunk_overlap: int = 150,
    ):
        # LLM
        self.llm = ChatOllama(
            model=llm_config["model"],
            temperature=llm_config.get("temperature", 0.2),
            base_url="http://127.0.0.1:11434"
        )

        # Embeddings
        self.embeddings = OllamaEmbeddings(model=embed_config["model"])

        # Indexing / retrieval state
        self.docs: List[Document] = []
        self.vectorstore: Optional[Chroma] = None
        self.persist_directory = persist_directory
        self.collection_name = collection_name

        # Splitter
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size, chunk_overlap=chunk_overlap
        )

    def _ensure_vectorstore(self):
        """Create or load a persistent Chroma store."""
        if self.vectorstore is not None:
            return

        if self.persist_directory:
            # Load existing collection if present, otherwise it will be created on first add
            self.vectorstore = Chroma(
                collection_name=self.collection_name,
                embedding_function=self.embeddings,
                persist_directory=self.persist_directory,
            )
        else:
            # In-memory (ephemeral) store
            self.vectorstore = Chroma(
                collection_name=self.collection_name,
                embedding_function=self.embeddings,
            )

    def load_documents(self, directory_path: str, glob_pattern: str = "*.pdf") -> int:
        """
        Load all PDFs from a directory, split, and add to Chroma.
        Returns the number of chunks added.
        """
        directory = Path(directory_path)
        if not directory.exists():
            raise FileNotFoundError(f"Directory not found: {directory_path}")

        all_docs: List[Document] = []
        for file_path in directory.glob(glob_pattern):
            try:
                loader = PyMuPDFLoader(str(file_path), mode="single")
                docs = loader.load()  # list[Document]
                all_docs.extend(docs)
            except Exception as e:
                print(f"Error processing {file_path.name}: {e}")

        if not all_docs:
            print("No documents loaded.")
            return 0

        # Save raw docs in memory (optional)
        self.docs.extend(all_docs)

        # Split and index
        chunks = self.text_splitter.split_documents(all_docs)
        if not chunks:
            print("No chunks produced after splitting.")
            return 0

        self._ensure_vectorstore()
        assert self.vectorstore is not None

        # To avoid dupes on re-ingestion, you can pass unique IDs (e.g., hash of text + source)
        # Here we keep it simple:
        self.vectorstore.add_documents(chunks)

        # Persist to disk if configured
        if self.persist_directory:
            self.vectorstore.persist()

        return len(chunks)

    def get_relevant_documents(self, query: str, k: int = 4) -> List[Document]:
        if self.vectorstore is None:
            raise ValueError("Vector store is not initialized. Load documents first.")
        return self.vectorstore.similarity_search(query, k=k)

    def _build_prompt(self, query: str, docs: List[Document]) -> str:
        context_sections = []
        for i, d in enumerate(docs, 1):
            src = d.metadata.get("source", "unknown")
            page = d.metadata.get("page", "n/a")
            context_sections.append(
                f"Source {i} | {src} | page {page}\n{d.page_content}"
            )
        context = "\n\n---\n\n".join(context_sections)
        return (
            "You are a helpful assistant. Use ONLY the context below to answer.\n"
            "If the answer is not in the context, say: 'Sorry, I do not know the answer.'\n\n"
            f"Context:\n{context}\n\n"
            f"Question: {query}\n\n"
            "Answer:"
        )

    def generate_response(self, query: str, k: int = 4) -> Tuple[str, List[dict]]:
        docs = self.get_relevant_documents(query, k=k)
        prompt = self._build_prompt(query, docs)
        result = self.llm.invoke(prompt)

        # Return answer + compact source info
        sources = [
            {
                "source": d.metadata.get("source", "unknown"),
                "page": d.metadata.get("page"),
                "score": None,  # similarity_search (not with scores). Switch to similarity_search_with_relevance_scores if needed.
            }
            for d in docs
        ]
        # `result` can be an AIMessage or string depending on LC version
        answer = getattr(result, "content", str(result))
        return answer, sources

In [5]:
rag = RAG(EVAL_LLM_CONFIG, EVAL_EMBED_CONFIG)
rag.load_documents(DIRECTORY_PATH)

C:\Users\Widi\AppData\Local\Temp\ipykernel_14556\1401042433.py:47: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  self.vectorstore = Chroma(
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


2

In [6]:
len(rag.docs)

1

In [7]:
# Load data
for doc in rag.docs:
    print(doc.metadata['source'])

C:\Users\Widi\rag\docs\test_tables\pdf_table1.pdf


In [8]:
# The model that will be used by RAGAS framework to evaluate LLM-based metrics
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0,
    max_tokens=None,
    top_p=0.95,
    top_k=40,)

In [9]:
# Asking a simple question to the RAG
start_time = time.time()
query = 'How many public schools are there in Abu Dhabi? Before you respond to my query please walk me through through your thought process step by step and tell me the sources that you use to get to the answer.'
relevant_doc = rag.get_relevant_documents(query)
answer = rag.generate_response(query)
end_time = time.time()

print(f"Query : {query}\n")
print(f"Answer : {answer[0]}\n")
print(f"Answer was taken from : {answer[1][0]['source']}\n")
print(f"Relevant document metadata: {relevant_doc[0].metadata}\n")
print(f"Relevant document page content: {relevant_doc[0].page_content}\n")
print(f"Elapsed time: {end_time - start_time:.4f} seconds")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Query : How many public schools are there in Abu Dhabi? Before you respond to my query please walk me through through your thought process step by step and tell me the sources that you use to get to the answer.

Answer : Okay, let's break this down.

1.  **Understanding the Question:** The question asks about the number of public schools in Abu Dhabi.

2.  **Analyzing the Context:** I have two PDF documents (Source 1 and Source 2) related to PIRLS assessments. These documents contain data on student performance in reading literacy assessments across various countries. However, they do not contain information about the number of public schools in Abu Dhabi.

3.  **Conclusion:** Based on the provided context, I do not have the information to answer the question about the number of public schools in Abu Dhabi.

Answer: Sorry, I do not know the answer.

Answer was taken from : C:\Users\Widi\rag\docs\test_tables\pdf_table1.pdf

Relevant document metadata: {'file_path': 'C:\\Users\\Widi\\rag

In [14]:
# Asking a simple question to the RAG
start_time = time.time()
query = 'How many schools were sampled for participation in PIRLS 2021? Before you respond to my query please walk me through through your thought process step by step.'
relevant_doc = rag.get_relevant_documents(query)
answer = rag.generate_response(query)
end_time = time.time()

print(f"Query : {query}\n")
print(f"Answer : {answer[0]}\n")
print(f"Answer was taken from : {answer[1][0]['source']}\n")
print(f"Relevant document metadata: {relevant_doc[0].metadata}\n")
print(f"Relevant document page content: {relevant_doc[0].page_content}\n")
print(f"Elapsed time: {end_time - start_time:.4f} seconds")

Query : How many schools were sampled for participation in PIRLS 2021? Before you respond to my query please walk me through through your thought process step by step.

Answer : Okay, let's break this down step by step to find the answer to the question: "How many schools were sampled for participation in PIRLS 2021?"

1. **Identify Relevant Sources:** I need to examine all the provided documents (Source 1, Source 2, Source 3, and Source 4) to find the information.

2. **Scan for Numbers:** I will carefully read each source, looking for any numbers related to the number of schools sampled for PIRLS 2021.

3. **Extract the Information:**
   * **Source 1 (Azerbaijan):** States "A total of 174 schools were sampled for participation in PIRLS 2021..."
   * **Source 2 (Austria):**  Doesn't explicitly state the number of schools sampled.
   * **Source 3 (Abu Dhabi UAE):** States "A total of 174 schools were sampled for participation in PIRLS 2021..."
   * **Source 4 (Germany):** Doesn't expli

In [13]:
# Asking a simple question to the RAG
start_time = time.time()
query = 'How many schools were sampled for participation in PIRLS 2021? Before you respond to my query please walk me through through your thought process step by step and tell me the sources that you use to get to the answer.'
relevant_doc = rag.get_relevant_documents(query)
answer = rag.generate_response(query)
end_time = time.time()

print(f"Query : {query}\n")
print(f"Answer : {answer[0]}\n")
print(f"Answer was taken from : {answer[1][0]['source']}\n")
print(f"Relevant document metadata: {relevant_doc[0].metadata}\n")
print(f"Relevant document page content: {relevant_doc[0].page_content}\n")
print(f"Elapsed time: {end_time - start_time:.4f} seconds")

Query : How many schools were sampled for participation in PIRLS 2021? Before you respond to my query please walk me through through your thought process step by step and tell me the sources that you use to get to the answer.

Answer : Okay, let’s break this down step-by-step to find the answer to the question: “How many schools were sampled for participation in PIRLS 2021?”

1. **Identify Relevant Sources:** I need to look through all the provided documents (Source 1, Source 2, Source 3, and Source 4) to find the number of schools sampled for PIRLS 2021.

2. **Analyze Source 1:**  Source 1 states: “A total of 174 schools were sampled for participation in PIRLS 2021, with approximately 8,500 students sampled.”

3. **Analyze Source 2:** Source 2 does not contain the number of schools sampled.

4. **Analyze Source 3:** Source 3 states: “A total of 174 schools were sampled for participation in PIRLS 2021, with approximately 8,500 students sampled.”

5. **Analyze Source 4:** Source 4 does 

In [15]:
# Asking a simple question to the RAG
start_time = time.time()
query = 'How many percent students started general education in private primary schools in 2018? Before you respond to my query please walk me through through your thought process step by step and tell me the sources that you use to get to the answer.'
relevant_doc = rag.get_relevant_documents(query)
answer = rag.generate_response(query)
end_time = time.time()

print(f"Query : {query}\n")
print(f"Answer : {answer[0]}\n")
print(f"Answer was taken from : {answer[1][0]['source']}\n")
print(f"Relevant document metadata: {relevant_doc[0].metadata}\n")
print(f"Relevant document page content: {relevant_doc[0].page_content}\n")
print(f"Elapsed time: {end_time - start_time:.4f} seconds")

Query : How many percent students started general education in private primary schools in 2018? Before you respond to my query please walk me through through your thought process step by step and tell me the sources that you use to get to the answer.

Answer : Okay, let's break this down step-by-step to find the answer.

1. **Identify the Question:** The question asks: "How many percent students started general education in private primary schools in 2018?"

2. **Locate Relevant Information:** I need to find the percentage of students who started general education in private primary schools in 2018. I will scan through the provided documents.

3. **Source Identification:**
   *   **Source 2:** C:\Users\Widi\rag\docs\test\Germany.pdf – This document discusses primary education in Germany, including the percentage of students attending private primary schools (3.6%).
   *   **Source 3:** C:\Users\Widi\rag\docs\test\Austria.pdf – This document does not contain the information requested.
 

In [29]:
start_time = time.time()
query = 'What is the total amount of tax breaks payments received by families as a part of Corona Aids?'
relevant_doc = rag.get_relevant_documents(query)
answer = rag.generate_response(query)
end_time = time.time()

print(f"Query : {query}\n")
print(f"Answer : {answer[0]}\n")
print(f"Answer was taken from : {answer[1][0]['source']}\n")
print(f"Relevant document metadata: {relevant_doc[0].metadata}\n")
print(f"Relevant document page content:\n{relevant_doc[0].page_content}\n")
print(f"Elapsed time: {end_time - start_time:.4f} seconds")

Query : What is the total amount of tax breaks payments received by families as a part of Corona Aids?

Answer : 2 billion euros.

Answer was taken from : C:\Users\Widi\rag\docs\test\Germany.pdf

Relevant document metadata: {'author': '', 'moddate': '2022-09-26T14:06:06-04:00', 'title': 'Germany.pdf', 'producer': 'Adobe PDF Library 21.1.174', 'trapped': '', 'creationdate': '2022-09-26T18:04:57+00:00', 'format': 'PDF 1.6', 'subject': '', 'source': 'C:\\Users\\Widi\\rag\\docs\\test\\Germany.pdf', 'keywords': '', 'file_path': 'C:\\Users\\Widi\\rag\\docs\\test\\Germany.pdf', 'total_pages': 16, 'creationDate': 'D:20220926180457Z', 'creator': 'Acrobat PDFMaker 21 for Word', 'modDate': "D:20220926140606-04'00'"}

Relevant document page content:
children and youth for the years 2021 and 2022 - 2 billion euros]. Retrieved from 
https://www.bmbf.de/bmbf/shareddocs/kurzmeldungen/de/kinder-und-jugendliche-nach-der-corona-
pandemie-staerken.html 
55  Federal Republic of Germany. (2020). Zweites Ges

In [30]:
# Retrieved contextsw
for i, doc in enumerate(relevant_doc):
    print(i, doc.metadata['source'])
    print()

0 C:\Users\Widi\rag\docs\test\Germany.pdf

1 C:\Users\Widi\rag\docs\test\Germany.pdf

2 C:\Users\Widi\rag\docs\test\Germany.pdf

3 C:\Users\Widi\rag\docs\test\Germany.pdf



In [31]:
relevant_doc[0]

Document(metadata={'author': '', 'moddate': '2022-09-26T14:06:06-04:00', 'title': 'Germany.pdf', 'producer': 'Adobe PDF Library 21.1.174', 'trapped': '', 'creationdate': '2022-09-26T18:04:57+00:00', 'format': 'PDF 1.6', 'subject': '', 'source': 'C:\\Users\\Widi\\rag\\docs\\test\\Germany.pdf', 'keywords': '', 'file_path': 'C:\\Users\\Widi\\rag\\docs\\test\\Germany.pdf', 'total_pages': 16, 'creationDate': 'D:20220926180457Z', 'creator': 'Acrobat PDFMaker 21 for Word', 'modDate': "D:20220926140606-04'00'"}, page_content='children and youth for the years 2021 and 2022 - 2 billion euros]. Retrieved from \nhttps://www.bmbf.de/bmbf/shareddocs/kurzmeldungen/de/kinder-und-jugendliche-nach-der-corona-\npandemie-staerken.html \n55  Federal Republic of Germany. (2020). Zweites Gesetz zur Umsetzung steuerlicher Hilfsmaßnahmen zur \nBewältigung der Corona-Krise (Zweites Corona-Steuerhilfegesetz) [Second Act Implementing Tax Relief \nMeasures to Address the Coronavirus Crisis (Second Coronavirus Tax 

In [32]:
len(relevant_doc)

4

In [33]:
for rel_doc in relevant_doc:
    print(rel_doc.metadata['source'])

C:\Users\Widi\rag\docs\test\Germany.pdf
C:\Users\Widi\rag\docs\test\Germany.pdf
C:\Users\Widi\rag\docs\test\Germany.pdf
C:\Users\Widi\rag\docs\test\Germany.pdf


In [34]:
relevant_doc[0].metadata['source']

'C:\\Users\\Widi\\rag\\docs\\test\\Germany.pdf'

# Evaluation

In [ ]:
# Asking the same quetion to Google Gemini
messages = [
    ("system", f"Context information is below.\n---\n{relevant_doc}\n---\nGiven the context information and not prior knowledge, answer the query. If you do not know the answer simply say 'I have no idea.'\nQuery: {query}\nAnswer:\n"),
    ("human", query),
]

llm.invoke(messages).content

In [36]:
# Gold Standard Dataset
# Document 1
# Question-Answer Pair #1
quest_1_doc_1 = 'How many public schools are there in Abu Dhabi?'
ans_1_doc_1 = '218'
ctx_1_doc_1 = '''
There are 218 public schools with a student population of 119,261, 80 percent of which are UAE nationals.
'''

# Question-Answer Pair #2
quest_2_doc_1 = 'What is the proportion of expats students in Abu Dhabi Private Schools?'
ans_2_doc_1 = '71%'
ctx_2_doc_1 = '''
"Exhibit 1: Demographics for Abu Dhabi Schools, Kindergarten/Foundation Stage 2* to Grade 12/Year 13
Indicators
Public Schools
Private Schools
Charter Schools
Number of Schools
218
203
22
Number of Students
119,261
246,752
30,397
Number of UAE Nationals
95,176 (80%)
72,075 (29%)
27,552 (91%)
Number of Expats
24,085 (20%)
174,677 (71%)
2,845 (9%)"
'''

# Question-Answer Pair #3
quest_3_doc_1 = 'How many hours that public school teachers were required to take as a minimum?'
ans_3_doc_1 = '25 hours'
ctx_3_doc_1 = '''
Public school teachers are required to undertake a minimum annual delivery of 25 hours of professional development. 
'''

# Question-Answer Pair #4
quest_4_doc_1 = 'What is Arabic Reading Challenge?'
ans_4_doc_1 = 'The Arab Reading Challenge (ARC) is an Arab literacy initiative designed to develop students’ fluency in Arabic and build their critical thinking and analytical skills.'
ctx_4_doc_1 = '''
The Arab Reading Challenge (ARC) is an Arab literacy initiative designed to develop students’ fluency in Arabic and build their critical thinking and analytical skills.
'''

# Question-Answer Pair #5
quest_5_doc_1 = 'How many schools were sampled for participation in PIRLS 2021?'
ans_5_doc_1 = '174 schools'
ctx_5_doc_1 = '''
A total of 174 schools were sampled for participation in PIRLS 2021, with approximately 8,500 students sampled. 
'''

# Question-Answer Pair #6
quest_6_doc_1 = 'When did Abu Dhabi first participated  as a benchmarking participant?'
ans_6_doc_1 = '2011'
ctx_6_doc_1 = '''
Abu Dhabi first participated in PIRLS in 2011 as a benchmarking participant, which was the first international reading performance benchmark for the Emirate. 
'''

# Question-Answer Pair #7
quest_7_doc_1 = 'How many different curricula offered in Abu Dhabi private school?'
ans_7_doc_1 = '14'
ctx_7_doc_1 = '''
Private schools, established and licensed across the Emirate of Abu Dhabi, currently offer 14 different curricula.
'''

# Question-Answer Pair #8
quest_8_doc_1 = 'Which cycle comprised to Grade 5 to 8 in Abu Dhabi public schools?'
ans_8_doc_1 = 'Cycle 2'
ctx_8_doc_1 = 'The MoE has developed a comprehensive United Arab Emirates school framework aligned to international standards named the Emirati School.1 In public schools, Cycle 1 comprises Grades 1 to 4, Cycle 2 comprises Grades 5 to 8, and Cycle 3 comprises Grades 9 to 12'

# Question-Answer Pair #9
quest_9_doc_1 = 'How many options are offered in the partial return models?'
ans_9_doc_1 = '4'
ctx_9_doc_1 = '''
"In planning for reopening, schools were asked to calculate their physical capacity; based on this capacity, each was able to choose its own reopening model from an option of full face-to-face (FTF) return or four partial return models. The partial models consisted of the following agile options in which any partial model was supplemented by distance learning:
• Half-day
• Alternating day
• Alternating week
• Hybrid alternating
'''

# Question-Answer Pair #10
quest_10_doc_1 = 'How many schools offers MoE curriculul in Abu Dhabi private schools?'
ans_10_doc_1 = '44'
ctx_10_doc_1 = '''
Exhibit 2: Curricula Offered in Abu Dhabi Private Schools
Curriculum
Number of Private Schools Offering the Curriculum
American
69
British
56
MoE
44
Indian
29
International Baccalaureate School (IBS)
12
SABISa
5
Canadian
3
French
3
Pakistani
3
Filipino
2
Bangladeshi
1
German
1
Japanese
1
Spanish
1
'''


In [37]:
# Gold Standard Dataset
# Document 2
# Question-Answer Pair #1
quest_1_doc_2 = 'How many major sectors are in the German education system?'
ans_1_doc_2 = '5'
ctx_1_doc_2 = 'The German education system consists of five major education sectors: preprimary, primary, lower secondary, upper secondary, and tertiary.'

# Question-Answer Pair #2
quest_2_doc_2 = 'How many time has Germany participated in PIRLS?'
ans_2_doc_2 = '5'
ctx_2_doc_2 = 'Germany has participated in PIRLS since the first study cycle in 2001, thus, for the fifth time in 2021.'

# Question-Answer Pair #3
quest_3_doc_2 = 'How many phases are in teacher education and professional development?'
ans_3_doc_2 = '3'
ctx_3_doc_2 = 'In general, teacher education and professional development is divided into three phases: university training, a one- to two-year supervised internship (depending on the federal state), and professional development in lifelong learning.'

# Question-Answer Pair #4
quest_4_doc_2 = 'What is the total amount of tax breaks payments received by families as a part of Corona Aids?'
ans_4_doc_2 = '450 euros'
ctx_4_doc_2 = 'The federal government provided financial “Corona Aids” for families, companies, and their employees. Families received tax breaks of three one-time payments totaling about 450 euros in 2020 and 2021.'

# Question-Answer Pair #5
quest_5_doc_2 = 'When was the Nobody Left Behind program first implemented in Schleswig-Holstein?'
ans_5_doc_2 = '2006'
ctx_5_doc_2 = 'The Nobody Left Behind program was first implemented in Schleswig-Holstein in 2006. Several expansions for different domains and school tracks followed. In academic year 2013–2014, the expansion Nobody Left Behind—Reading Makes You Strong! Primary School was introduced in 100 primary schools in Schleswig-Holstein and has since been rolled out in Lower Saxony as well.'

# Question-Answer Pair #6
quest_6_doc_2 = 'In which grade does the Secondary education starts in Berlin?'
ans_6_doc_2 = '5th grade'
ctx_6_doc_2 = 'Secondary education starts with fifth grade (seventh grade in Berlin and Brandenburg) and lasts between five and nine years (four to seven years in Berlin and Brandenburg) or until the age of 18 (depending on the federal state).'

# Question-Answer Pair #7
quest_7_doc_2 = 'How many percent students started general education in private primary schools in 2018?'
ctx_7_doc_2 = 'Only 3.6 percent (in 2018) of students started general education in private primary schools.'
ans_7_doc_2 = '3.6 percent'

# Question-Answer Pair #8
quest_8_doc_2 = 'How many primary schools participated in 2013-2014 Nobody Left Behind program in Schleswig-Holstein?'
ans_8_doc_2 = '100'
ctx_8_doc_2 = 'In academic year 2013–2014, the expansion Nobody Left Behind—Reading Makes You Strong! Primary School was introduced in 100 primary schools in Schleswig-Holstein and has since been rolled out in Lower Saxony as well.'


# Question-Answer Pair #9
quest_9_doc_2 = 'In which semester of second grade of primary school does the earliest grading start?'
ans_9_doc_2 = '2nd'
ctx_9_doc_2 = 'The earliest grading in primary school starts in the second semester of second grade in some federal states, but this policy varies.'

# Question-Answer Pair #10
quest_10_doc_2 = 'What was the rate of 7-days-incidence exceeded to which caused school closure between April - June 2021?'
ans_10_doc_2 = '165 per 100,000 people'
ctx_10_doc_2 = 'However, between April 24, 2021, and June 30, 2021, the national emergency brake act required school closures when the local 7-days-incidence exceeded 165 per 100,000 people.'

In [38]:
test_queries = [
    quest_1_doc_1, quest_2_doc_1, quest_3_doc_1, quest_4_doc_1, quest_5_doc_1, quest_6_doc_1, quest_7_doc_1, quest_8_doc_1, quest_9_doc_1, quest_10_doc_1,
    quest_1_doc_2, quest_2_doc_2, quest_3_doc_2, quest_4_doc_2, quest_5_doc_2, quest_6_doc_2, quest_7_doc_2, quest_8_doc_2, quest_9_doc_2, quest_10_doc_2]

expected_responses = [
    ans_1_doc_1, ans_2_doc_1, ans_3_doc_1, ans_4_doc_1, ans_5_doc_1, ans_6_doc_1, ans_7_doc_1, ans_8_doc_1, ans_9_doc_1, ans_10_doc_1,
    ans_1_doc_2, ans_2_doc_2, ans_3_doc_2, ans_4_doc_2, ans_5_doc_2, ans_6_doc_2, ans_7_doc_2, ans_8_doc_2, ans_9_doc_2, ans_10_doc_2,]

relevant_documents = [
    ctx_1_doc_1, ctx_2_doc_1, ctx_3_doc_1, ctx_4_doc_1, ctx_5_doc_1, ctx_6_doc_1, ctx_7_doc_1, ctx_8_doc_1, ctx_9_doc_1, ctx_10_doc_1,
    ctx_1_doc_2, ctx_2_doc_2, ctx_3_doc_2, ctx_4_doc_2, ctx_5_doc_2, ctx_6_doc_2, ctx_7_doc_2, ctx_8_doc_2, ctx_9_doc_2, ctx_10_doc_2,]


In [39]:
start_time = time.time()
dataset = []

for query, response in zip(test_queries, expected_responses):
    relevant_doc = rag.get_relevant_documents(query)
    answer = rag.generate_response(query)

    dataset.append({
        "user_input" : query,
        "retrieved_contexts" : [relevant_doc[0].metadata['source']], # Get the top retrieved context
        "response" : answer[0],
        "reference" : response
    })

end_time = time.time()
print(f"Elapsed time: {end_time - start_time:.4f} seconds")

Elapsed time: 20.5815 seconds


In [40]:
df = pd.DataFrame(dataset)
df

# Expected responses are named as reference because LLMContextPrecisionWithReference() requires input that has a column called reference

,user_input,retrieved_contexts,response,reference
0,How many public schools are there in Abu Dhabi?,[C:\Users\Widi\rag\docs\test\Abu Dhabi UAE.pdf],There are 218 public schools in Abu Dhabi.,218
1,What is the proportion of expats students in A...,[C:\Users\Widi\rag\docs\test\Abu Dhabi UAE.pdf],71%,71%
2,How many hours that public school teachers wer...,[C:\Users\Widi\rag\docs\test\Abu Dhabi UAE.pdf],25 hours,25 hours
3,What is Arabic Reading Challenge?,[C:\Users\Widi\rag\docs\test\Abu Dhabi UAE.pdf],The Arab Reading Challenge (ARC) is an Arab li...,The Arab Reading Challenge (ARC) is an Arab li...
4,How many schools were sampled for participatio...,[C:\Users\Widi\rag\docs\test\Azerbaijan.pdf],200,174 schools
5,When did Abu Dhabi first participated as a be...,[C:\Users\Widi\rag\docs\test\Abu Dhabi UAE.pdf],In 2011.,2011
6,How many different curricula offered in Abu Dh...,[C:\Users\Widi\rag\docs\test\Abu Dhabi UAE.pdf],There are 14 different curricula offered in Ab...,14
7,Which cycle comprised to Grade 5 to 8 in Abu D...,[C:\Users\Widi\rag\docs\test\Abu Dhabi UAE.pdf],Cycle 2 comprises Grades 5 to 8.,Cycle 2
8,How many options are offered in the partial re...,[C:\Users\Widi\rag\docs\test\Abu Dhabi UAE.pdf],Four,4
9,How many schools offers MoE curriculul in Abu ...,[C:\Users\Widi\rag\docs\test\Abu Dhabi UAE.pdf],"Sorry, I do not know the answer.",44


In [41]:
df.iloc[4]['response']

'200'

In [42]:
evaluation = EvaluationDataset.from_pandas(df)


In [43]:
evaluator_llm = LangchainLLMWrapper(llm)

result_base = evaluate(evaluation, [LLMContextRecall(),
                               Faithfulness(),
                               FactualCorrectness(),
                               LLMContextPrecisionWithReference()],
                  evaluator_llm)
result_base

Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

{'context_recall': 0.0500, 'faithfulness': 0.0000, 'factual_correctness(mode=f1)': 0.3810, 'llm_context_precision_with_reference': 0.0000}

In [44]:
dir(result_base)

['__annotations__',
 '__class__',
 '__dataclass_fields__',
 '__dataclass_params__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__match_args__',
 '__module__',
 '__ne__',
 '__new__',
 '__post_init__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_repr_dict',
 '_scores_dict',
 'binary_columns',
 'cost_cb',
 'dataset',
 'ragas_traces',
 'run_id',
 'scores',
 'to_pandas',
 'total_cost',
 'total_tokens',
 'traces',
 'upload']

In [45]:
result_base.to_pandas()

,user_input,retrieved_contexts,response,reference,context_recall,faithfulness,factual_correctness(mode=f1),llm_context_precision_with_reference
0,How many public schools are there in Abu Dhabi?,[C:\Users\Widi\rag\docs\test\Abu Dhabi UAE.pdf],There are 218 public schools in Abu Dhabi.,218,0.0,0.0,0.00,0.0
1,What is the proportion of expats students in A...,[C:\Users\Widi\rag\docs\test\Abu Dhabi UAE.pdf],71%,71%,0.0,0.0,1.00,0.0
2,How many hours that public school teachers wer...,[C:\Users\Widi\rag\docs\test\Abu Dhabi UAE.pdf],25 hours,25 hours,0.0,0.0,1.00,0.0
3,What is Arabic Reading Challenge?,[C:\Users\Widi\rag\docs\test\Abu Dhabi UAE.pdf],The Arab Reading Challenge (ARC) is an Arab li...,The Arab Reading Challenge (ARC) is an Arab li...,1.0,0.0,1.00,0.0
4,How many schools were sampled for participatio...,[C:\Users\Widi\rag\docs\test\Azerbaijan.pdf],200,174 schools,0.0,0.0,0.00,0.0
5,When did Abu Dhabi first participated as a be...,[C:\Users\Widi\rag\docs\test\Abu Dhabi UAE.pdf],In 2011.,2011,0.0,0.0,1.00,0.0
6,How many different curricula offered in Abu Dh...,[C:\Users\Widi\rag\docs\test\Abu Dhabi UAE.pdf],There are 14 different curricula offered in Ab...,14,0.0,0.0,0.00,0.0
7,Which cycle comprised to Grade 5 to 8 in Abu D...,[C:\Users\Widi\rag\docs\test\Abu Dhabi UAE.pdf],Cycle 2 comprises Grades 5 to 8.,Cycle 2,0.0,0.0,0.00,0.0
8,How many options are offered in the partial re...,[C:\Users\Widi\rag\docs\test\Abu Dhabi UAE.pdf],Four,4,0.0,0.0,1.00,0.0
9,How many schools offers MoE curriculul in Abu ...,[C:\Users\Widi\rag\docs\test\Abu Dhabi UAE.pdf],"Sorry, I do not know the answer.",44,0.0,0.0,0.00,0.0


In [16]:
# Asking a simple question to the RAG
start_time = time.time()
query = 'What was the rate of 7-days-incidence exceeded to which caused school closure between April - June 2021? Before you respond to my query please walk me through through your thought process step by step and tell me the sources that you use to get to the answer.'
relevant_doc = rag.get_relevant_documents(query)
answer = rag.generate_response(query)
end_time = time.time()

print(f"Query : {query}\n")
print(f"Answer : {answer[0]}\n")
print(f"Answer was taken from : {answer[1][0]['source']}\n")
print(f"Relevant document metadata: {relevant_doc[0].metadata}\n")
print(f"Relevant document page content: {relevant_doc[0].page_content}\n")
print(f"Elapsed time: {end_time - start_time:.4f} seconds")

Query : What was the rate of 7-days-incidence exceeded to which caused school closure between April - June 2021? Before you respond to my query please walk me through through your thought process step by step and tell me the sources that you use to get to the answer.

Answer : Okay, let's break this down step-by-step to find the answer.

1. **Understanding the Question:** The question asks about the specific incidence rate (7-days-incidence) that triggered school closures between April - June 2021. This means we need to find a source that details the criteria for school closures in Germany during that timeframe.

2. **Analyzing Source 3:**  I’ll start by examining Source 3, which discusses Germany’s response to the pandemic. The key phrase is “national emergency brake act required school closures when the local 7-days-incidence exceeded 165 per 100,000 people.”

3. **Extracting the Relevant Information:** This sentence directly states that a 7-days-incidence exceeding 165 per 100,000 p

In [10]:
# Asking a simple question to the RAG
start_time = time.time()
query = 'Are there tables in the source files? Before you respond to my query please walk me through through your thought process step by step.'
relevant_doc = rag.get_relevant_documents(query)
answer = rag.generate_response(query)
end_time = time.time()

print(f"Query : {query}\n")
print(f"Answer : {answer[0]}\n")
print(f"Answer was taken from : {answer[1][0]['source']}\n")
print(f"Relevant document metadata: {relevant_doc[0].metadata}\n")
print(f"Relevant document page content: {relevant_doc[0].page_content}\n")
print(f"Elapsed time: {end_time - start_time:.4f} seconds")

Query : Are there tables in the source files? Before you respond to my query please walk me through through your thought process step by step.

Answer : Okay, let's break this down step-by-step to determine if there are tables within the provided source files.

1. **Understanding the Context:** I'm given four source files (Table 1, Table 2, Table 2, and Table 2). Each source contains data related to the PIRLS 2021 assessment.  Table 2 appears multiple times, which is a key detail.

2. **Identifying Tables:** The prompt asks if there *are* tables in the source files.  The source files themselves *do* contain tables. Table 1, Table 2, and Table 2 are all tables.

3. **Confirmation:**  Based on the document descriptions, each source contains data presented in tabular format.

Therefore, the answer is yes, there are tables in the source files.

**Final Answer:** Yes


Answer was taken from : C:\Users\Widi\rag\docs\test_tables\pdf_table1.pdf

Relevant document metadata: {'format': 'PDF 1.6'

In [27]:
# Asking a simple question to the RAG
start_time = time.time()
query = 'If there is a table in the source file, tell me what is the title? Before you respond to my query please walk me through through your thought process step by step.'
relevant_doc = rag.get_relevant_documents(query)
answer = rag.generate_response(query)
end_time = time.time()

print(f"Query : {query}\n")
print(f"Answer : {answer[0]}\n")
print(f"Answer was taken from : {answer[1][0]['source']}\n")
print(f"Relevant document metadata: {relevant_doc[0].metadata}\n")
print(f"Relevant document page content: {relevant_doc[0].page_content}\n")
print(f"Elapsed time: {end_time - start_time:.4f} seconds")

Query : If there is a table in the source file, tell me what is the title? Before you respond to my query please walk me through through your thought process step by step.

Answer : Okay, let's break this down step by step.

1.  **Understand the Question:** The question asks for the title of a table present in the provided source files.

2.  **Analyze the Context:** I have two source files (C:\Users\Widi\rag\docs\test_tables\pdf_table1.pdf). Both files contain tables.

3.  **Identify the Relevant Table:** The first source file (Exhibit 4.3) describes trends in the distribution of students reaching PIRLS benchmarks. The second source file contains a table with "Prior to COVID-19 pandemic" and "During COVID-19 pandemic" labels.

Therefore, the title of the table is: "Prior to COVID-19 pandemic        During COVID-19 pandemic".


Answer was taken from : C:\Users\Widi\rag\docs\test_tables\pdf_table1.pdf

Relevant document metadata: {'file_path': 'C:\\Users\\Widi\\rag\\docs\\test_tables\\pd

In [28]:
# Asking a simple question to the RAG
start_time = time.time()
query = 'If there is a table in the source file, tell me what is the first text in the table? Before you respond to my query please walk me through through your thought process step by step.'
relevant_doc = rag.get_relevant_documents(query)
answer = rag.generate_response(query)
end_time = time.time()

print(f"Query : {query}\n")
print(f"Answer : {answer[0]}\n")
print(f"Answer was taken from : {answer[1][0]['source']}\n")
print(f"Relevant document metadata: {relevant_doc[0].metadata}\n")
print(f"Relevant document page content: {relevant_doc[0].page_content}\n")
print(f"Elapsed time: {end_time - start_time:.4f} seconds")

Query : If there is a table in the source file, tell me what is the first text in the table? Before you respond to my query please walk me through through your thought process step by step.

Answer : Okay, let's break this down step by step.

1.  **Understand the Question:** The question asks for the first text within a table described in the provided source files.

2.  **Analyze the Context:** I have two source files. Source 1 describes Exhibit 4.3, which is a table showing trends in PIRLS International Benchmarks. Source 2 describes a table with data related to student performance before and during the COVID-19 pandemic.

3.  **Identify the Relevant Table:** The question refers to "a table in the source file." Since Source 1 describes Exhibit 4.3, I will examine that table.

4.  **Locate the First Text:** Looking at the text provided for Source 1, the first text in the table described as Exhibit 4.3 is: “2001”.

Therefore, the answer is: 2001

Answer was taken from : C:\Users\Widi\ra

In [ ]:
# Asking a simple question to the RAG
start_time = time.time()
query = 'What are the ? Before you respond to my query please walk me through through your thought process step by step.'
relevant_doc = rag.get_relevant_documents(query)
answer = rag.generate_response(query)
end_time = time.time()

print(f"Query : {query}\n")
print(f"Answer : {answer[0]}\n")
print(f"Answer was taken from : {answer[1][0]['source']}\n")
print(f"Relevant document metadata: {relevant_doc[0].metadata}\n")
print(f"Relevant document page content: {relevant_doc[0].page_content}\n")
print(f"Elapsed time: {end_time - start_time:.4f} seconds")